# 03b_segger_to_common: segger output -> common segmentation format.

```text
Reads segger_segmentation.parquet (one row per transcript), keeps the
recommended assignments (`filtered` == True), and writes the 10x-style matrix
+ cells.csv.gz + assignment_summary.csv that R/helpers.R::load_segmentation()
reads, with core ids from 02_tma_dearray.R.

Cell centroid = mean position of the cell's transcripts. Cell area comes from
`segger export boundaries` (GeoParquet) if available, else it is left NA
(03c then skips the area-based metrics for segger instead of guessing).
```

**Notebook version of `experiment1_tma/03b_segger_to_common.py`.** Same code, laid out for interactive work:

1. Set the options in the **Parameters** cell (the script's command-line options: `--slide-id` becomes `slide_id`).
2. Run the cells in order. After each cell you can inspect its variables, tables and images.

Use the `.py` script for batch jobs on the cluster; both produce the same outputs. This notebook is generated by `tools/make_notebooks.py`, which overwrites it, so save your own edits under another name.

In [ ]:
import os

# folder of 03b_segger_to_common.py; this notebook sits in notebooks/ next to it
SCRIPT_DIR = os.path.abspath('..')

# Which Python runs this notebook? It must be the conda environment, not the system one.
import sys
print('Python:', sys.executable)
try:
    import numpy  # noqa: F401
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'This kernel is not the exp1-imaging environment. In a terminal, once:\n'
        '  conda activate exp1-imaging\n'
        '  python -m ipykernel install --user --name exp1-imaging --display-name exp1-imaging\n'
        'then in Jupyter: Kernel > Change kernel > exp1-imaging  (environment file: experiment1_tma/env/py_imaging.yml)')

# Packages this step needs (import name: pip name). Anything missing is installed INTO THIS
# kernel with %pip in a notebook cell -- not with pip in a terminal, which may be another Python.
import importlib.util
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'pyarrow': 'pyarrow'}
missing = [pip for mod, pip in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError(
        f'Missing in this kernel: {", ".join(missing)}\n'
        f'Run this in a new cell:  %pip install {" ".join(missing)}\n'
        'then Kernel > Restart, and run the notebook again from the top.')
print('All packages found.')

In [ ]:
import argparse
import glob
import os
import sys

import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, os.path.join(SCRIPT_DIR, "py"))
from xenium_io import Transcripts, read_features, write_common_format  # noqa: E402

## Parameters

In [ ]:
from types import SimpleNamespace

# Same options as `python experiment1_tma/03b_segger_to_common.py --help` (--slide-id -> slide_id).

# Options marked REQUIRED have no default: replace their None, e.g. slide_id="TMA_ORGAN".
# Paths are plain strings in quotes.

a = SimpleNamespace(
    slide_id=None,        # REQUIRED
    xenium_dir=None,      # REQUIRED
    segger_parquet=None,  # REQUIRED. segger_segmentation.parquet written by segger, or the folder that contains it
    boundaries=None,
    cores_csv=None,       # REQUIRED
    out_dir=None,         # REQUIRED
    min_transcripts=1,
    min_qv=20.0,
)

missing = [k for k in ('slide_id', 'xenium_dir', 'segger_parquet', 'cores_csv', 'out_dir') if getattr(a, k) in (None, '')]
if missing:
    raise ValueError(f'Fill in these options in the cell above (replace None with a value): {missing}')

In [ ]:
# Options not set in the Parameters cell (e.g. a cell copied from an older version
# of this notebook) get the script's default. Nothing to edit here.
_defaults = dict(
    slide_id=None,
    xenium_dir=None,
    segger_parquet=None,
    boundaries=None,
    cores_csv=None,
    out_dir=None,
    min_transcripts=1,
    min_qv=20.0,
)
for _k, _v in _defaults.items():
    if not hasattr(a, _k):
        setattr(a, _k, _v)
        print(f'{_k} not in the Parameters cell -> default {_v!r}')

## Helper functions

In [ ]:
def core_of(x, y, cores):
    """Core id for points by core bounding box (first match), NaN outside."""
    out = np.full(len(x), None, dtype=object)
    for c in cores.itertuples(index=False):
        m = (x >= c.xmin) & (x < c.xmax) & (y >= c.ymin) & (y < c.ymax) & (out == None)  # noqa: E711
        out[m] = c.core_id
    return out

In [ ]:
SEGGER_FILE = "segger_segmentation.parquet"

In [ ]:
def find_segger_parquet(path):
    """The segger output file: a path to it, or a folder containing it (searched recursively)."""
    p = os.path.expanduser(path)
    hint = ""
    if not os.path.isabs(p) and os.path.isdir(os.sep + p.split(os.sep)[0]):      # e.g. "work_ikmb/..."
        hint = f"\n  The path is relative -- did you mean '/{p}' (leading slash missing)?"
    if os.path.isfile(p):
        return p
    if os.path.isdir(p):
        hits = sorted(glob.glob(os.path.join(p, "**", SEGGER_FILE), recursive=True))
        if len(hits) == 1:
            print(f"segger output found: {hits[0]}")
            return hits[0]
        if len(hits) > 1:
            raise FileNotFoundError(f"several {SEGGER_FILE} under {p} -- give the full path of one:\n  "
                                    + "\n  ".join(hits))
        found = sorted(os.listdir(p))[:20]
        raise FileNotFoundError(
            f"no {SEGGER_FILE} under {p}. It is written by segger itself (03b_segment_segger.sh, "
            f"step 'segger segment'), so segger has probably not run (or not finished) for this slide.\n"
            f"  Folder contents: {found}")
    raise FileNotFoundError(
        f"{p} does not exist.{hint}\n  Expected <project>/data/segmentation/segger_raw/<slide_id>/{SEGGER_FILE}, "
        f"written by 03b_segment_segger.sh (segger must have run first).")

## Run

In [ ]:
a.segger_parquet = find_segger_parquet(a.segger_parquet)
features = read_features(a.xenium_dir)
name_to_idx = {n: i for i, n in enumerate(features["name"])}
gene_names = set(features.loc[features["type"] == "Gene Expression", "name"])

In [ ]:
seg = pd.read_parquet(a.segger_parquet,
                      columns=["segger_cell_id", "x", "y", "feature_name", "filtered"])
seg["feature_name"] = seg["feature_name"].astype(str)
print(f"{len(seg):,} transcripts in segger output")

In [ ]:
cores = pd.read_csv(a.cores_csv)
cores = cores[(cores["slide_id"] == a.slide_id) & cores["core_id"].notna()]

keep = seg["filtered"].fillna(False).astype(bool) & seg["segger_cell_id"].notna()
asg = seg.loc[keep].copy()
asg["cell"] = asg["segger_cell_id"].astype(str)
asg["feat"] = asg["feature_name"].map(name_to_idx)
asg = asg[asg["feat"].notna()]
asg["feat"] = asg["feat"].astype(np.int64)

In [ ]:
cells = (asg.groupby("cell", sort=False)
            .agg(x_centroid=("x", "mean"), y_centroid=("y", "mean"), n_tx=("x", "size"))
            .reset_index())
cells = cells[cells["n_tx"] >= a.min_transcripts].reset_index(drop=True)
cells["core_id"] = core_of(cells["x_centroid"].to_numpy(), cells["y_centroid"].to_numpy(), cores)
cells = cells[cells["core_id"].notna()].reset_index(drop=True)
cell_index = pd.Series(np.arange(len(cells)), index=cells["cell"])

In [ ]:
asg = asg[asg["cell"].isin(cell_index.index)]
rows = cell_index.loc[asg["cell"]].to_numpy()
counts = sp.coo_matrix((np.ones(len(rows), dtype=np.int32), (rows, asg["feat"].to_numpy())),
                       shape=(len(cells), len(features))).tocsr()
counts.sum_duplicates()

In [ ]:
cells["cell_area"] = np.nan
cells["nucleus_area"] = np.nan
if a.boundaries and os.path.exists(a.boundaries):
    try:
        import geopandas as gpd
        b = gpd.read_parquet(a.boundaries)
        id_col = next(c for c in b.columns if c != b.geometry.name)
        area = pd.Series(b.geometry.area.to_numpy(), index=b[id_col].astype(str))
        cells["cell_area"] = cells["cell"].map(area)
        print(f"cell areas from {a.boundaries} (id column '{id_col}')")
    except Exception as e:  # keep going without areas
        print(f"WARNING: could not read boundaries ({e}); cell_area left NA")

In [ ]:
# Assignment summary on the same transcript definition as 03a: qv-filtered
# gene transcripts inside each core box. 10x assignment comes from the raw
# transcripts table; segger's from its own output.
tx = Transcripts(a.xenium_dir, features, min_qv=a.min_qv)
seg_gene = seg[seg["feature_name"].isin(gene_names)]
seg_core = core_of(seg_gene["x"].to_numpy(), seg_gene["y"].to_numpy(), cores)
seg_ok = keep.loc[seg_gene.index].to_numpy()
summaries = []
for c in cores.itertuples(index=False):
    idx = tx.window(c.xmin, c.xmax, c.ymin, c.ymax)
    g = tx.is_gene[idx]
    in_core = seg_core == c.core_id
    summaries.append({
        "slide_id": a.slide_id, "region_id": c.core_id,
        "n_tx_genes": int(g.sum()),
        # segger may keep low-qv transcripts, so its rate uses its own denominator
        "n_tx_genes_segger": int(in_core.sum()),
        "n_assigned_segger": int((in_core & seg_ok).sum()),
        "n_assigned_xenium": int((g & tx.xen_assigned[idx]).sum()),
    })

In [ ]:
cells["cell_id"] = cells["cell"]
cells["segmentation_method"] = "segger"
write_common_format(a.out_dir, counts, cells["cell_id"].to_numpy(), features,
                    cells[["cell_id", "core_id", "x_centroid", "y_centroid", "cell_area",
                           "nucleus_area", "segmentation_method"]],
                    pd.DataFrame(summaries))